# 09 Retrieval eval
Measure recall at k for four retrievers at two chunk sizes, and check citation faithfulness.

Step 1: set up the kit

In [1]:
# Setup: in Colab, clone the kit first. Locally, find the kit folder. Change REPO_URL to your fork.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/arpitRanjan/atp-builder-kit"
if "google.colab" in sys.modules and not Path("kit/llm.py").exists():
    if not Path("atp-builder-kit").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("atp-builder-kit")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rank_bm25==0.2.2", "google-genai==3.3.0", "openai==3.51.0"], check=True)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "kit" / "llm.py").exists())
sys.path.insert(0, str(ROOT))
CORPUS = Path(os.environ.get("CORPUS_DIR", ROOT / "corpus"))
OUTPUT = Path(os.environ.get("KIT_OUTPUT_DIR", ROOT / "stage-09"))
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Kit folder found:", ROOT.name)

Kit folder found: atp-builder-kit


Step 2: load the labels and the pages

In [2]:
import json, random, re, time
import pandas as pd
from kit.llm import call_llm, default_model, pick_provider
from kit.prices import PRICES, cost
from kit.rag import (BM25Index, VectorIndex, chunk_pages, get_embedder, get_reranker, parse_corpus, rerank, rrf)

pages = parse_corpus(CORPUS)
questions = [json.loads(line) for line in open(CORPUS / "questions.jsonl")]
embedder, reranker = get_embedder(), get_reranker()
print(len(questions), "labeled questions | embedder:", embedder.name, "| reranker:", reranker.name)

25 labeled questions | embedder: hash-384 | reranker: overlap-stand-in


Step 3: recall at k, computed in code

In [3]:
def recall_at_k(hits, page_ids, k):
    # Share of the labeled pages found in the top k chunks.
    found = {chunk["page_id"] for chunk, _ in hits[:k]}
    return len(found & set(page_ids)) / len(page_ids)

print(recall_at_k([({"page_id": "qn-02"}, 1.0)], ["qn-02", "qn-05"], 1))

0.5


Step 4: four retrievers: keyword, vector, hybrid, hybrid plus reranker

In [4]:
def make_retrievers(chunks):
    vector, keyword = VectorIndex(chunks, embedder), BM25Index(chunks)
    hybrid = lambda q, k: rrf([vector.search(q, 20), keyword.search(q, 20)], k)
    # The three strategies to compare, plus keyword only (bm25) as a reference row.
    return {"vector": vector.search, "hybrid": hybrid,
            "hybrid_rerank": lambda q, k: rerank(reranker, q, hybrid(q, 20), k), "bm25 (reference)": keyword.search}

Step 5: score them at one chunk size, then a changed one

In [5]:
MODEL = default_model(pick_provider())
price_in = PRICES.get(MODEL, {"input": 0})["input"]

def evaluate(size, overlap):
    chunks = chunk_pages(pages, size, overlap)
    rows = []
    for name, search in make_retrievers(chunks).items():
        recalls, latency, tokens = {1: [], 3: [], 5: []}, [], []
        for q in questions:
            start = time.time()
            hits = search(q["question"], 5)
            latency.append(time.time() - start)
            tokens.append(sum(len(c["text"]) for c, _ in hits) / 4)  # what the top 5 adds to the answer prompt
            for k in recalls:
                recalls[k].append(recall_at_k(hits, q["page_ids"], k))
        rows.append({"chunk_size": size, "retriever": name, **{f"recall@{k}": sum(v) / len(v) for k, v in recalls.items()},
                     "ms_per_query": 1000 * sum(latency) / len(latency), "search_cost_usd": 0.0,
                     "answer_input_cost_usd": sum(tokens) / len(tokens) * price_in / 1e6})
    return rows

table = pd.DataFrame(evaluate(120, 30) + evaluate(40, 10))  # one chunk size, then one change
table.round(5).to_csv(OUTPUT / "retrieval_strategies.csv", index=False)  # recall at k per strategy and size
table.round(5)

,chunk_size,retriever,recall@1,recall@3,recall@5,ms_per_query,search_cost_usd,answer_input_cost_usd
0,120,vector,0.74,0.92,0.94,0.03914,0.0,0.00075
1,120,hybrid,0.78,0.94,0.98,0.11040,0.0,0.00075
2,120,hybrid_rerank,0.70,0.94,0.98,0.54789,0.0,0.00071
3,120,bm25 (reference),0.78,0.90,0.98,0.06151,0.0,0.00074
4,40,vector,0.62,0.90,0.98,0.04792,0.0,0.00027
5,40,hybrid,0.68,0.96,0.98,0.17087,0.0,0.00027
6,40,hybrid_rerank,0.62,0.84,0.88,0.35235,0.0,0.00026
7,40,bm25 (reference),0.72,0.94,0.98,0.10094,0.0,0.00027


Step 6: answers with citations, some of them planted wrong

In [6]:
SYSTEM = ("Answer using only the numbered sources. Cite the source ids you used in square brackets, "
          "like [qn-02#0]. If the sources do not answer the question, say so.")
chunks = chunk_pages(pages, 120, 30)
vector = VectorIndex(chunks, embedder)
rng = random.Random(7)
cases = []
for n, q in enumerate(questions):
    hits = vector.search(q["question"], 3)
    r = call_llm("Sources:\n" + "\n".join(f"[{c['id']}] {c['text']}" for c, _ in hits) + f"\n\nQuestion: {q['question']}",
                 system=SYSTEM, max_output_tokens=1000)
    ids = re.findall(r"qn-\d+#\d+", r["text"])
    cited = next((c for c in chunks if ids and c["id"] == ids[0]), hits[0][0])
    if n % 2:  # plant a wrong citation: swap in a chunk from another page
        cited = rng.choice([c for c in chunks if c["page_id"] != cited["page_id"]])
    cases.append({"qid": q["qid"], "question": q["question"], "answer": r["text"], "cited": cited["id"],
                  "source": cited["text"], "label_faithful": cited["page_id"] in q["page_ids"]})
cases = pd.DataFrame(cases)
print(cases.label_faithful.value_counts().to_dict())

{False: 15, True: 10}


Step 7: a small citation judge, checked against the labels

In [7]:
JUDGE = ("Does the source support the answer to the question? The source must contain the facts the "
         "answer gives for this question. Reply SUPPORTED or NOT SUPPORTED.")

def judge(row):
    r = call_llm(f"Question: {row.question}\nAnswer: {row.answer}\nSource: {row.source}\nReply with one verdict.",
                 system=JUDGE, max_output_tokens=800)
    return not r["text"].strip().upper().startswith("NOT")

cases["judge_faithful"] = [judge(row) for row in cases.itertuples()]
print("Citation faithfulness, judge:", round(cases.judge_faithful.mean(), 2), "| labels:", round(cases.label_faithful.mean(), 2))
print(pd.crosstab(cases.label_faithful, cases.judge_faithful, rownames=["label"], colnames=["judge"]))
cases[["qid", "cited", "label_faithful", "judge_faithful", "answer"]].to_csv(OUTPUT / "citation_faithfulness.csv", index=False)

Citation faithfulness, judge: 0.56 | labels: 0.4
judge  False  True 
label              
False     11      4
True       0     10


Step 8: the same check in CI. `evals/retrieval_check.py` runs recall at k, citation faithfulness and the leak test, and exits 1 under the bar

In [8]:
import subprocess
CHOSEN = "hybrid"  # set this to the setup you chose from the table in step 5, and use the same in CI
run = subprocess.run([sys.executable, str(ROOT / "evals" / "retrieval_check.py"), "--strategy", CHOSEN,
                      "--out", str(OUTPUT)], capture_output=True, text=True, cwd=ROOT)
print(run.stdout or run.stderr)
print("Exit code:", run.returncode, "| saved retrieval_eval.csv, retrieval_summary.json and leak_test.json")

25 labeled questions | hybrid, 120 word chunks
Recall@5: 0.98 (bar 0.8) | citation faithfulness: 0.88 (bar 0.7)
Leak test, ada (admin): cited ['qn-13#0'], admin only chunks retrieved: ['qn-13#0', 'qn-13#1', 'qn-14#0', 'qn-14#1', 'qn-15#0', 'qn-15#1']
Leak test, pat (public): cited nothing, admin only chunks retrieved: none
Leak test: passed | overall: passed

Exit code: 0 | saved retrieval_eval.csv, retrieval_summary.json and leak_test.json
